In [1]:
# Download data dan src dari GitHub (uncomment jika di Colab)
!rm -rf IR-BERT-Transformer data src
!git clone https://github.com/teranixbq/IR-BERT-Transformer.git
!mv IR-BERT-Transformer/data ./data
!mv IR-BERT-Transformer/src ./src
!rm -rf IR-BERT-Transformer

Cloning into 'IR-BERT-Transformer'...
remote: Enumerating objects: 105, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 105 (delta 50), reused 62 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (105/105), 187.00 KiB | 7.48 MiB/s, done.
Resolving deltas: 100% (50/50), done.


# Neural IR Exercise dengan BERT Cross-Encoder
Dosen: Zico Pratama Putra
Kelompok: [Nama 1] - [Nama 2] - [Nama 3]
Tanggal: ---

# **A. Setup**

In [2]:
from src.judgement_aggregation import load_raw_judgements, aggregate_judgements, save_qrels, compute_annotator_reliability
from src.bert_cross_encoder import BERTCrossEncoder
from transformers import pipeline
import pandas as pd

DEST = "/content/data/"
RAW_JUDGEMENTS = DEST + "fira_raw_judgements.tsv"
BASELINE_2022 = DEST + "fira-2022.baseline-qrels.tsv"

# **B. Load Raw**

In [3]:
df = load_raw_judgements(RAW_JUDGEMENTS)

Loaded 237 raw judgements
Columns: ['query_id', 'doc_id', 'judgement', 'confidence', 'annotator_id', 'duration_ms']
   query_id doc_id  judgement  confidence annotator_id  duration_ms
0         1   d1_1          3        0.85       User_5        30289
1         1   d1_1          3        0.94       User_0        83482
2         1   d1_1          3        0.92       User_4        16165
3         1   d1_2          2        0.81       User_0        38062
4         1   d1_2          1        0.92       User_5        12851


# **C. Helper Function**

## C.1 For Part 1

In [4]:
def compare_with_baseline(
    aggregated_df: pd.DataFrame,
    baseline_qrels: pd.DataFrame
):
    """
    Membandingkan advanced aggregation dengan baseline qrels.

    Metrik:
    - matched query-doc pairs
    - exact match
    - exact match rate
    - mean absolute difference
    """

    merged = aggregated_df.merge(
        baseline_qrels[["query_id", "doc_id", "baseline_score"]],
        on=["query_id", "doc_id"],
        how="inner"
    )

    merged["score_diff"] = (
        merged["score"] -
        merged["baseline_score"]
    )

    merged["abs_diff"] = merged["score_diff"].abs()

    total_pairs = len(merged)
    exact_match = (merged["score_diff"] == 0).sum()

    exact_match_rate = (
        exact_match / total_pairs
        if total_pairs > 0
        else 0
    )

    mean_abs_diff = (
        merged["abs_diff"].mean()
        if total_pairs > 0
        else 0
    )

    print("=== Comparison with Baseline Qrels ===")
    print(f"Matched query-doc pairs  : {total_pairs}")
    print(f"Exact match              : {exact_match}")
    print(f"Exact match rate         : {exact_match_rate:.4f}")
    print(f"Mean absolute difference : {mean_abs_diff:.4f}")

    print("\nDistribusi perbedaan score:")
    print(merged["score_diff"].value_counts().sort_index())

    return merged

## C.2 For Part 2

In [5]:
# Metric Functions: MRR@10, NDCG@10, Precision@10

def mrr_at_k(relevances, k=10):
    """
    MRR@K untuk satu query.
    """

    relevances = relevances[:k]

    for idx, rel in enumerate(relevances):
        if rel > 0:
            return 1.0 / (idx + 1)

    return 0.0


def dcg_at_k(relevances, k=10):
    """
    DCG@K.
    """

    relevances = np.asarray(relevances[:k])

    if len(relevances) == 0:
        return 0.0

    gains = (2 ** relevances - 1)
    discounts = np.log2(np.arange(len(relevances)) + 2)

    return np.sum(gains / discounts)


def ndcg_at_k(relevances, k=10):
    """
    NDCG@K.
    """

    dcg = dcg_at_k(relevances, k)
    ideal = dcg_at_k(sorted(relevances, reverse=True), k)

    if ideal == 0:
        return 0.0

    return dcg / ideal


def precision_at_k(relevances, k=10):
    """
    Precision@K.
    """

    relevances = relevances[:k]

    if len(relevances) == 0:
        return 0.0

    binary_rels = [
        1 if rel > 0 else 0
        for rel in relevances
    ]

    return sum(binary_rels) / k

# **PART 1 : FiRA Judgement Aggregation**

## 1.1 Baseline - Simple Majority Vote

In [6]:
agg_maj = aggregate_judgements(df, method='majority')
agg_maj.head(20)

Aggregated into 79 unique query-doc pairs


,query_id,doc_id,score,num_judgements,mean_score,median_score,std_score
0,1,d1_1,3,3,3.000000,3.0,0.000000
1,1,d1_2,2,3,1.666667,2.0,0.471405
2,1,d1_3,1,3,1.333333,1.0,0.471405
3,1,d1_4,0,3,0.000000,0.0,0.000000
4,1,d1_5,0,3,0.000000,0.0,0.000000
5,1,d1_6,0,3,0.000000,0.0,0.000000
6,1,d1_7,1,3,1.000000,1.0,0.000000
7,1,d1_8,1,3,0.666667,1.0,0.471405
8,2,d2_1,3,3,2.666667,3.0,0.471405
9,2,d2_2,2,3,2.333333,2.0,0.471405


## 1.2 Confidence-Based Weighting
Bobot berdasarkan seberapa yakin annotator

In [7]:
agg_weighted = aggregate_judgements(df, method='advanced', strategy='confidence_weighted')
agg_weighted.head(20)

Aggregated into 79 unique query-doc pairs


,query_id,doc_id,score,num_judgements,mean_score,median_score,std_score
0,1,d1_1,3,3,3.000000,3.0,0.000000
1,1,d1_2,2,3,1.666667,2.0,0.471405
2,1,d1_3,1,3,1.333333,1.0,0.471405
3,1,d1_4,0,3,0.000000,0.0,0.000000
4,1,d1_5,0,3,0.000000,0.0,0.000000
5,1,d1_6,0,3,0.000000,0.0,0.000000
6,1,d1_7,1,3,1.000000,1.0,0.000000
7,1,d1_8,1,3,0.666667,1.0,0.471405
8,2,d2_1,3,3,2.666667,3.0,0.471405
9,2,d2_2,2,3,2.333333,2.0,0.471405


## 1.3 Reliability-Weighted Voting
Bobot dari track record annotator — seberapa sering ia setuju dengan suara terbanyak (majority vote). Annotator yang sering benar dapat bobot lebih besar.

In [8]:
reliability = compute_annotator_reliability(df)
reliability

{'User_0': 0.7692307692307693,
 'User_1': 0.8372093023255814,
 'User_2': 0.7804878048780488,
 'User_3': 0.7659574468085106,
 'User_4': 0.8275862068965517,
 'User_5': 0.7631578947368421}

In [9]:
agg_reliability = aggregate_judgements(df, method='advanced', strategy='annotator_reliability', annotator_reliability=reliability)
agg_reliability.head(20)

Aggregated into 79 unique query-doc pairs


,query_id,doc_id,score,num_judgements,mean_score,median_score,std_score
0,1,d1_1,3,3,3.000000,3.0,0.000000
1,1,d1_2,2,3,1.666667,2.0,0.471405
2,1,d1_3,1,3,1.333333,1.0,0.471405
3,1,d1_4,0,3,0.000000,0.0,0.000000
4,1,d1_5,0,3,0.000000,0.0,0.000000
5,1,d1_6,0,3,0.000000,0.0,0.000000
6,1,d1_7,1,3,1.000000,1.0,0.000000
7,1,d1_8,1,3,0.666667,1.0,0.471405
8,2,d2_1,3,3,2.666667,3.0,0.471405
9,2,d2_2,2,3,2.333333,2.0,0.471405


## 1.4 Bandingkan Majority Dengan Advanced

### 1.4.1 Majority & Advanced (Confidence-Based Weighting)

In [10]:
comparison_df1 = agg_maj.merge(
    agg_weighted,
    on=["query_id", "doc_id"],
    suffixes=("_majority", "_advanced")
)

comparison_df1["score_diff"] = (
    comparison_df1["score_advanced"] -
    comparison_df1["score_majority"]
)

print("Jumlah query-doc pair:", len(comparison_df1))
print("Jumlah score berbeda :", (comparison_df1["score_diff"] != 0).sum())

print("\nDistribusi perbedaan score:")
print(comparison_df1["score_diff"].value_counts().sort_index())

display(comparison_df1.head())

Jumlah query-doc pair: 79
Jumlah score berbeda : 6

Distribusi perbedaan score:
score_diff
-1     4
 0    73
 1     2
Name: count, dtype: int64


,query_id,doc_id,score_majority,num_judgements_majority,mean_score_majority,median_score_majority,std_score_majority,score_advanced,num_judgements_advanced,mean_score_advanced,median_score_advanced,std_score_advanced,score_diff
0,1,d1_1,3,3,3.000000,3.0,0.000000,3,3,3.000000,3.0,0.000000,0
1,1,d1_2,2,3,1.666667,2.0,0.471405,2,3,1.666667,2.0,0.471405,0
2,1,d1_3,1,3,1.333333,1.0,0.471405,1,3,1.333333,1.0,0.471405,0
3,1,d1_4,0,3,0.000000,0.0,0.000000,0,3,0.000000,0.0,0.000000,0
4,1,d1_5,0,3,0.000000,0.0,0.000000,0,3,0.000000,0.0,0.000000,0


### 1.4.2 Majority & Advanced (Reliability-Weighted Voting)

In [11]:
comparison_df2 = agg_maj.merge(
    agg_reliability,
    on=["query_id", "doc_id"],
    suffixes=("_majority", "_advanced")
)

comparison_df2["score_diff"] = (
    comparison_df2["score_advanced"] -
    comparison_df2["score_majority"]
)

print("Jumlah query-doc pair:", len(comparison_df2))
print("Jumlah score berbeda :", (comparison_df2["score_diff"] != 0).sum())

print("\nDistribusi perbedaan score:")
print(comparison_df2["score_diff"].value_counts().sort_index())

display(comparison_df2.head())

Jumlah query-doc pair: 79
Jumlah score berbeda : 6

Distribusi perbedaan score:
score_diff
-1     4
 0    73
 1     2
Name: count, dtype: int64


,query_id,doc_id,score_majority,num_judgements_majority,mean_score_majority,median_score_majority,std_score_majority,score_advanced,num_judgements_advanced,mean_score_advanced,median_score_advanced,std_score_advanced,score_diff
0,1,d1_1,3,3,3.000000,3.0,0.000000,3,3,3.000000,3.0,0.000000,0
1,1,d1_2,2,3,1.666667,2.0,0.471405,2,3,1.666667,2.0,0.471405,0
2,1,d1_3,1,3,1.333333,1.0,0.471405,1,3,1.333333,1.0,0.471405,0
3,1,d1_4,0,3,0.000000,0.0,0.000000,0,3,0.000000,0.0,0.000000,0
4,1,d1_5,0,3,0.000000,0.0,0.000000,0,3,0.000000,0.0,0.000000,0


## 1.5 Bandingkan BASELINE dengan Advanced

In [12]:
baseline_df = pd.read_csv(BASELINE_2022, sep=r"\s+", names=["query_id", "unused", "doc_id", "baseline_score"])

### 1.5.1 BASELINE & Advanced (Confidence - Based Weighting)

In [13]:
if BASELINE_2022 is not None:
    baseline_comparison_df = compare_with_baseline(
        agg_weighted,
        baseline_df
    )

    display(baseline_comparison_df.head())
else:
    print("Baseline comparison dilewati karena baseline qrels tidak tersedia.")

=== Comparison with Baseline Qrels ===
Matched query-doc pairs  : 79
Exact match              : 73
Exact match rate         : 0.9241
Mean absolute difference : 0.0759

Distribusi perbedaan score:
score_diff
-1     4
 0    73
 1     2
Name: count, dtype: int64


,query_id,doc_id,score,num_judgements,mean_score,median_score,std_score,baseline_score,score_diff,abs_diff
0,1,d1_1,3,3,3.000000,3.0,0.000000,3,0,0
1,1,d1_2,2,3,1.666667,2.0,0.471405,2,0,0
2,1,d1_3,1,3,1.333333,1.0,0.471405,1,0,0
3,1,d1_4,0,3,0.000000,0.0,0.000000,0,0,0
4,1,d1_5,0,3,0.000000,0.0,0.000000,0,0,0


### 1.5.2 BASELINE & Advanced (Reliability-Weighted Voting)

In [14]:
if BASELINE_2022 is not None:
    baseline_comparison_df = compare_with_baseline(
        agg_reliability,
        baseline_df
    )

    display(baseline_comparison_df.head())
else:
    print("Baseline comparison dilewati karena baseline qrels tidak tersedia.")

=== Comparison with Baseline Qrels ===
Matched query-doc pairs  : 79
Exact match              : 73
Exact match rate         : 0.9241
Mean absolute difference : 0.0759

Distribusi perbedaan score:
score_diff
-1     4
 0    73
 1     2
Name: count, dtype: int64


,query_id,doc_id,score,num_judgements,mean_score,median_score,std_score,baseline_score,score_diff,abs_diff
0,1,d1_1,3,3,3.000000,3.0,0.000000,3,0,0
1,1,d1_2,2,3,1.666667,2.0,0.471405,2,0,0
2,1,d1_3,1,3,1.333333,1.0,0.471405,1,0,0
3,1,d1_4,0,3,0.000000,0.0,0.000000,0,0,0
4,1,d1_5,0,3,0.000000,0.0,0.000000,0,0,0


## 1.7 Analisis Manual Disagreement Tinggi

In [15]:
def inspect_high_disagreement_cases(
    raw_df: pd.DataFrame,
    aggregated_df: pd.DataFrame,
    top_n=5
):
    """
    Menampilkan contoh query-doc pair dengan disagreement tertinggi.

    Tujuan:
    - memeriksa variasi judgement antar annotator,
    - melihat apakah final score advanced aggregation masuk akal,
    - digunakan untuk analisis laporan.
    """

    high_disagreement = aggregated_df.sort_values(
        by="std_score",
        ascending=False
    ).head(top_n)

    for _, row in high_disagreement.iterrows():
        qid = row["query_id"]
        did = row["doc_id"]

        group = raw_df[
            (raw_df["query_id"] == qid) &
            (raw_df["doc_id"].astype(str) == str(did))
        ]

        print("=" * 80)
        print(f"Query ID             : {qid}")
        print(f"Doc ID               : {did}")
        print(f"Aggregated score     : {row['score']}")
        print(f"Number of judgements : {row['num_judgements']}")
        print(f"Std score            : {row['std_score']:.2f}")
        print("\nRaw judgements:")
        display(group)

In [16]:
inspect_high_disagreement_cases(
    raw_df=df,
    aggregated_df=agg_weighted,
    top_n=5
)

Query ID             : 9
Doc ID               : d9_3
Aggregated score     : 1
Number of judgements : 3
Std score            : 0.94

Raw judgements:


,query_id,doc_id,judgement,confidence,annotator_id,duration_ms
198,9,d9_3,0,0.67,User_2,43300
199,9,d9_3,0,0.96,User_5,17720
200,9,d9_3,2,0.64,User_3,61968


Query ID             : 6
Doc ID               : d6_7
Aggregated score     : 1
Number of judgements : 3
Std score            : 0.94

Raw judgements:


,query_id,doc_id,judgement,confidence,annotator_id,duration_ms
138,6,d6_7,0,0.97,User_1,89656
139,6,d6_7,2,0.85,User_4,23359
140,6,d6_7,2,0.55,User_2,72910


Query ID             : 4
Doc ID               : d4_7
Aggregated score     : 1
Number of judgements : 3
Std score            : 0.94

Raw judgements:


,query_id,doc_id,judgement,confidence,annotator_id,duration_ms
90,4,d4_7,0,0.93,User_5,55279
91,4,d4_7,2,0.82,User_1,51363
92,4,d4_7,2,0.87,User_2,50752


Query ID             : 9
Doc ID               : d9_2
Aggregated score     : 2
Number of judgements : 3
Std score            : 0.94

Raw judgements:


,query_id,doc_id,judgement,confidence,annotator_id,duration_ms
195,9,d9_2,3,0.67,User_5,56548
196,9,d9_2,3,0.56,User_1,51132
197,9,d9_2,1,0.59,User_0,75483


Query ID             : 6
Doc ID               : d6_2
Aggregated score     : 2
Number of judgements : 3
Std score            : 0.82

Raw judgements:


,query_id,doc_id,judgement,confidence,annotator_id,duration_ms
123,6,d6_2,1,0.95,User_3,61692
124,6,d6_2,3,0.67,User_2,45065
125,6,d6_2,2,0.95,User_1,18826


## 1.6 Simpan Semua

In [17]:
save_qrels(agg_maj, DEST + "fira_aggregated.qrels")
save_qrels(agg_weighted, DEST + "fira_aggregated_confidence_weighted.qrels")
save_qrels(agg_reliability, DEST + "fira_aggregated_annotator_reliability.qrels")

Qrels saved to /content/data/fira_aggregated.qrels
Qrels saved to /content/data/fira_aggregated_confidence_weighted.qrels
Qrels saved to /content/data/fira_aggregated_annotator_reliability.qrels


# **Part 2: BERT Cross-Encoder Re-Ranking**

## 2.1 Setup Dataframe

In [18]:
df_tuples = pd.read_csv(DEST + "fira-2022.tuples.tsv", sep="\t",
                        names=["query_id", "doc_id", "query", "passage"])

# Skip kolom ke-2 (0/col), ambil qid, doc, score saja
qrels = pd.read_csv(DEST + "fira_aggregated.qrels", sep=r"\s+",
                    names=["qid", "doc", "score"], usecols=[0, 2, 3])

df = df_tuples.merge(qrels, left_on=["query_id", "doc_id"], right_on=["qid", "doc"])
df = df.drop(columns=["qid", "doc"])

df["label"] = (df["score"] >= 2).astype(int)

print(f"Total pairs: {len(df)}")
print(f"Relevant: {df['label'].sum()}, Irrelevant: {(1-df['label']).sum()}")
df.head()

Total pairs: 79
Relevant: 21, Irrelevant: 58


,query_id,doc_id,query,passage,score,label
0,1,d1_1,how to make a good cappuccino,"To make a perfect cappuccino, you need equal p...",3,1
1,1,d1_2,how to make a good cappuccino,Making a cappuccino at home requires an espres...,2,1
2,1,d1_3,how to make a good cappuccino,Cappuccino is an espresso-based coffee drink t...,1,0
3,1,d1_4,how to make a good cappuccino,Coffee beans are roasted to develop their flav...,0,0
4,1,d1_5,how to make a good cappuccino,The history of tea dates back to ancient China...,0,0


## 2.2 Train

In [19]:
reranker = BERTCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
reranker.train(df[["query", "passage", "label"]], epochs=5)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ Model cross-encoder/ms-marco-MiniLM-L-6-v2 loaded on cuda
Epoch 1/5 — avg loss: 0.7858
Epoch 2/5 — avg loss: 0.4409
Epoch 3/5 — avg loss: 0.3271
Epoch 4/5 — avg loss: 0.2714
Epoch 5/5 — avg loss: 0.2111


## 2.3 Rerank All Queries

In [20]:
results = []
for (qid, q), group in df.groupby(["query_id", "query"]):
    passages = group["passage"].tolist()
    doc_ids = group["doc_id"].tolist()
    labels = group["label"].tolist()

    ranked_idx, scores = reranker.re_rank(q, passages)

    ranked_docs = [doc_ids[i] for i in ranked_idx]
    ranked_scores = [scores[i] for i in ranked_idx]
    ranked_labels = [labels[i] for i in ranked_idx]

    results.append({
        "query_id": qid,
        "query": q,
        "ranked_docs": ranked_docs,
        "scores": ranked_scores,
        "labels": ranked_labels,
    })

    print(f"Query {qid}")
    for rank, (doc, sc, lb) in enumerate(zip(ranked_docs, ranked_scores, ranked_labels), 1):
        print(f"  {rank}. {doc} (score={sc:.4f}, relevant={lb})")
    print()

Re-ranking: 100%|██████████| 1/1 [00:00<00:00, 22.15it/s]


Query 1
  1. d1_1 (score=0.9999, relevant=1)
  2. d1_2 (score=0.9996, relevant=1)
  3. d1_3 (score=0.9109, relevant=0)
  4. d1_7 (score=0.0126, relevant=0)
  5. d1_6 (score=0.0000, relevant=0)
  6. d1_4 (score=0.0000, relevant=0)
  7. d1_5 (score=0.0000, relevant=0)
  8. d1_8 (score=0.0000, relevant=0)



Re-ranking: 100%|██████████| 1/1 [00:00<00:00, 22.58it/s]


Query 2
  1. d2_2 (score=0.9995, relevant=1)
  2. d2_1 (score=0.9992, relevant=1)
  3. d2_5 (score=0.9510, relevant=0)
  4. d2_3 (score=0.6694, relevant=0)
  5. d2_4 (score=0.0127, relevant=0)
  6. d2_8 (score=0.0000, relevant=0)
  7. d2_6 (score=0.0000, relevant=0)
  8. d2_7 (score=0.0000, relevant=0)



Re-ranking: 100%|██████████| 1/1 [00:00<00:00, 22.91it/s]


...

Query 8
  1. d8_1 (score=0.9998, relevant=1)
  2. d8_7 (score=0.9993, relevant=0)
  3. d8_2 (score=0.9967, relevant=1)
  4. d8_5 (score=0.3696, relevant=0)
  5. d8_4 (score=0.0485, relevant=0)
  6. d8_3 (score=0.0022, relevant=0)
  7. d8_6 (score=0.0002, relevant=0)
  8. d8_8 (score=0.0000, relevant=0)



Re-ranking: 100%|██████████| 1/1 [00:00<00:00, 23.42it/s]


Query 9
  1. d9_1 (score=0.9999, relevant=1)
  2. d9_2 (score=0.9999, relevant=1)
  3. d9_7 (score=0.9956, relevant=0)
  4. d9_3 (score=0.9077, relevant=0)
  5. d9_4 (score=0.8998, relevant=0)
  6. d9_6 (score=0.7861, relevant=0)
  7. d9_5 (score=0.0649, relevant=0)
  8. d9_8 (score=0.0020, relevant=0)



Re-ranking: 100%|██████████| 1/1 [00:00<00:00, 26.44it/s]

Query 10
  1. d10_1 (score=1.0000, relevant=1)
  2. d10_7 (score=0.9989, relevant=1)
  3. d10_2 (score=0.9980, relevant=1)
  4. d10_3 (score=0.0062, relevant=0)
  5. d10_4 (score=0.0014, relevant=0)
  6. d10_8 (score=0.0004, relevant=0)
  7. d10_5 (score=0.0002, relevant=0)



In [ ]:
reranker = BERTCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")

query = "How to make a good cappuccino?"
passages = ["To make a perfect cappuccino, you need equal parts of espresso, steamed milk, and milk foam. Start by pulling a double shot of espresso into your cup. Next, steam your milk to create a velvety microfoam. Pour the steamed milk over the espresso, then spoon the thick foam on top. Garnish with a sprinkle of cocoa powder if desired.", "coffe", "coffe"]
ranked_idx, scores = reranker.re_rank(query, passages)
print("Ranked order:", ranked_idx)

## 2.3 Evaluasi Performa Reranker

### 2.3.1 Evaluasi Dengan MS MARCO Dev/Test set

### 2.3.2 Evaluasi Dengan Data FiRA

# **Part 3: Extractive QA**

In [ ]:
qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2")

result = qa_pipeline(question=query, context=passages[ranked_idx[0]])
print(result)

## Evaluasi
# TODO: Implement evaluasi MRR@10, NDCG@10, dll.